# 4. Which model predicts a repeat purchase within 90 days?

Each row is a purchase. At the time of that purchase, we predict whether the same customer will buy again within 90 days. We compare eight classifiers on the same purchases, first with transaction information and then with four additional customer attributes.

[Data](#data) · [Time windows](#windows) · [Models and metric](#models) · [Starting results](#results) · [Tuning](#tuning) · [Kaggle submission](#kaggle-rf)

<a id="data"></a>

## 4.1 Preparing the purchase data

Notebook 01 identified 65 rows with neither a date nor transaction predictors. We exclude those rows and apply its historical corrections to the remaining purchases. The shared cleaning function uses counts and amounts from the same row to recover exact values. It leaves unresolved inconsistencies missing. We keep the purchase ID for submission, the customer ID for joining tables and the date for splitting by time.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

root = Path.cwd()
if not (root / 'data').is_dir():
    root = root.parent
sys.path.insert(0, str(root))
from src.history_cleaning import repair_history

target = 'repeat_purchase_90d'
raw = pd.read_csv(root / 'data' / 'train.csv')
clients = pd.read_csv(root / 'data' / 'clientes.csv')
dates = pd.to_datetime(raw['purchase_date'], errors='raise')
predictors = raw.columns.drop(['ID', 'customer_id', target, 'purchase_date'])
empty_rows = raw[predictors].isna().all(axis=1)
assert empty_rows.equals(dates.isna())
assert raw['ID'].is_unique and raw[target].isin([0, 1]).all()

data, audit = repair_history(raw.loc[dates.notna()].copy())
data['purchase_date'] = dates.loc[data.index]
data = data.reset_index(drop=True)
print('Eligible purchases:', len(data))
print('Purchases with inconsistent historical values:', audit['Distinct rows with invalid values'])

Eligible purchases: 6469
Purchases with inconsistent historical values: 227


### Recovering missing time intervals

This additional rule assumes that tenure starts at the first purchase and that `prior_mean_gap_days` averages the gaps **between previous purchases**. Three previous purchases give two such gaps. In one record, tenure is 133 days and the two gaps average 14.5 days. The last previous purchase was therefore 133 − 2 × 14.5 = 104 days ago.

If either recency or the mean gap is known, we can recover the other when at least two prior purchases exist. With one prior purchase, recency equals tenure and the mean gap is undefined. The code checks the identity wherever both quantities are known. Rounding recency removes floating-point error because purchase dates have day precision.

In [2]:
def recover_time_intervals(frame):
    count = frame['prior_purchase_count']
    tenure = frame['customer_tenure_days']
    recency = frame['days_since_previous_purchase']
    mean_gap = frame['prior_mean_gap_days']

    known_recency = (tenure - (count - 1) * mean_gap).where(count >= 2)
    known_recency = known_recency.mask(count == 1, tenure)
    known_gap = (tenure - recency) / (count - 1).where(count >= 2)

    for observed, derived in [(recency, known_recency), (mean_gap, known_gap)]:
        overlap = observed.notna() & derived.notna()
        assert (observed[overlap] - derived[overlap]).abs().lt(1e-6).all()
    return pd.DataFrame({
        'days_since_previous_purchase': recency.fillna(known_recency.round()),
        'prior_mean_gap_days': mean_gap.fillna(known_gap),
    })

time_columns = ['days_since_previous_purchase', 'prior_mean_gap_days']
original_times = data[time_columns].copy()
data[time_columns] = recover_time_intervals(data)
recovered = original_times.isna().sum() - data[time_columns].isna().sum()
display(recovered.rename('Recovered values').to_frame())

,Recovered values
days_since_previous_purchase,201
prior_mean_gap_days,203


In [3]:
client_features = ['country_or_region', 'commercial_region', 'island', 'customer_since_year']

def add_client_features(transactions):
    assert transactions['customer_id'].isin(clients['customer_id']).all()
    joined = transactions.merge(clients[['customer_id'] + client_features],
                                on='customer_id', how='left', validate='many_to_one', sort=False)
    assert joined['ID'].equals(transactions['ID'])
    joined['customer_since_year'] = joined['customer_since_year'].astype('string')
    return joined

purchases = add_client_features(data)
transaction_features = predictors.drop('random_noise').tolist()
feature_sets = {
    'Transactions': transaction_features,
    'Transactions + clients': transaction_features + client_features,
}
y = purchases[target]
print('Transaction inputs:', len(transaction_features))
print('Extra customer inputs:', len(client_features))

Transaction inputs: 34
Extra customer inputs: 4


The join must keep exactly one row per purchase in its original order. Entry year is a category so that each year can have its own effect. Using these four attributes assumes that their recorded values were available at purchase time. The client file has no dated versions to verify changes in region or island.

Notebook 03 tested `random_noise` and its final logistic selector discarded it. We omit it from both input sets. We keep the other transaction variables so that a feature discarded by logistic regression can still contribute to a tree split or a neural network. The same purchases enter both comparisons.

<a id="windows"></a>

## 4.2 Training on the past and validating on later purchases

We keep the main split from Notebook 03. The purchase at the 70% position in date order falls on 13 July 2022. A purchase contributes to training only if its entire 90-day outcome was observable before the validation period began. This leaves 4,424 purchases for model development, 103 in the gap and 1,942 in the later block. Notebook 03 already evaluated logistic regression on that later block.

Inside the 4,424 purchases, we move the prediction date forward three times. The first two validation windows cover a year each. The third ends at the original training cutoff. These dates give three consecutive periods of comparable size. For every model we use the same boundaries and a gap of 90 **calendar days**. Earlier validation purchases can enter a later training set once their outcomes are known. Customers can appear again because the task includes returning customers.

In [4]:
date = purchases['purchase_date']
horizon = pd.Timedelta(days=90)
main_start = date.sort_values().iloc[int(len(date) * 0.70)]
development_end = main_start - horizon
boundaries = [pd.Timestamp('2019-07-01'), pd.Timestamp('2020-07-01'),
              pd.Timestamp('2021-07-01'), development_end]

folds = []
periods = []
for start, end in zip(boundaries, boundaries[1:]):
    train_rows = date < start - horizon
    gap_rows = (date >= start - horizon) & (date < start)
    valid_rows = (date >= start) & (date < end)
    assert date[train_rows].max() + horizon < date[valid_rows].min()
    assert date[valid_rows].max() < development_end
    assert y[train_rows].nunique() == 2 and y[valid_rows].nunique() == 2
    folds.append((train_rows, valid_rows))
    periods.append([date[train_rows].max().date(), start.date(), (end - pd.Timedelta(days=1)).date(),
                    train_rows.sum(), gap_rows.sum(), valid_rows.sum(), y[valid_rows].mean()])
display(pd.DataFrame(periods, columns=['Last training date', 'Validation from', 'Through', 'Train', 'Gap',
                                      'Validation', 'Repeat rate']).round(3))

,Last training date,Validation from,Through,Train,Gap,Validation,Repeat rate
0,2019-04-01,2019-07-01,2020-06-30,2287,86,707,0.546
1,2020-03-31,2020-07-01,2021-06-30,2962,118,692,0.595
2,2021-03-30,2021-07-01,2022-04-13,3649,123,652,0.514


<a id="models"></a>

## 4.3 Models, preprocessing and score

In each training window we learn numerical medians, missing-value indicators and scaling. For categories we fill missing entries with `Missing` and learn one-hot columns. An unseen category can still be transformed. Scaling matters for KNN distances and neural network optimisation. We keep the encoded table dense so Gaussian Naive Bayes can accept it too. Scaling also lets us use the same preparation across all families.

We measure **F1 for repeat purchase**, the competition metric. Precision asks how many predicted returners came back. Recall asks how many actual returners we found. F1 gives both a role. We compute it separately for each period and average the three scores equally, so one larger period cannot dominate. Predicting a return for every purchase is our reference.

We apply thresholds of 0.50 and 0.45 to the same probabilities. At 0.45, an extra purchase counts as positive when its probability falls between these two cutoffs. Whether that helps F1 depends on how many of those extra positives actually returned.

In [5]:
numeric = make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler())
categorical = make_pipeline(SimpleImputer(strategy='constant', fill_value='Missing'),
                            OneHotEncoder(handle_unknown='ignore', sparse_output=False))
preparation = ColumnTransformer([
    ('numbers', numeric, make_column_selector(dtype_include='number')),
    ('categories', categorical, make_column_selector(dtype_exclude='number')),
])

thresholds = [0.50, 0.45]
window_names = ['Period 1', 'Period 2', 'Period 3']

def period_scores(classifier, inputs):
    scores = {threshold: [] for threshold in thresholds}
    for train_rows, valid_rows in folds:
        model = make_pipeline(clone(preparation), clone(classifier))
        model.fit(inputs.loc[train_rows], y.loc[train_rows])
        probability = model.predict_proba(inputs.loc[valid_rows])[:, 1]
        for threshold in thresholds:
            predicted = probability >= threshold
            scores[threshold].append(f1_score(y.loc[valid_rows], predicted))
    return scores

### What each model adds to the comparison

Logistic regression provides a linear reference. Gaussian Naive Bayes models variables separately within each class. KNN predicts from nearby purchases. A decision tree learns conditional splits. A random forest averages trees, while gradient boosting builds trees in sequence to correct earlier predictions.

In Week 11, histogram gradient boosting groups numeric values into bins before choosing tree splits. We start with 100 boosting iterations and up to 31 leaves per tree. In Weeks 12 and 13, the MLP learns a nonlinear relationship across inputs. We use one hidden layer with 16 units, ReLU activation and the Adam optimiser. This keeps the network small for training windows of about 2,300 to 3,650 purchases. Its 1,000 iterations are an upper limit. Both new models use a fixed seed and disable validation-based early stopping so the calendar windows control evaluation. One seed makes the run reproducible but cannot measure seed-to-seed variation.

In [6]:
models = {
    'Logistic regression': LogisticRegression(max_iter=10000),
    'Gaussian Naive Bayes': GaussianNB(),
    'K nearest neighbours': KNeighborsClassifier(n_neighbors=5),
    'Decision tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.10, random_state=42),
    'Histogram gradient boosting': HistGradientBoostingClassifier(
        max_iter=100, max_leaf_nodes=31, early_stopping=False, random_state=42),
    'Small MLP': MLPClassifier(hidden_layer_sizes=(16,), activation='relu', solver='adam',
                               alpha=0.0001, max_iter=1000, early_stopping=False, random_state=42),
}

### Does the timing correction improve prediction?

First compare logistic regression with and without the two timing corrections. The folds, inputs and 0.50 threshold are fixed. This isolates the effect of changing those two columns, including their remaining missing values.

In [7]:
before = purchases[transaction_features].copy()
before[time_columns] = original_times
logistic = models['Logistic regression']
timing_check = pd.DataFrame([
    period_scores(logistic, before)[0.50],
    period_scores(logistic, purchases[transaction_features])[0.50],
], columns=window_names, index=['Notebook 01 rules', 'With timing recovery'])
timing_check['Mean F1'] = timing_check.mean(axis=1)
display(timing_check.round(4))

,Period 1,Period 2,Period 3,Mean F1
Notebook 01 rules,0.7233,0.7190,0.7108,0.7177
With timing recovery,0.7180,0.7206,0.7083,0.7157


Recovering the time intervals raised logistic F1 in period 2 and reduced it in periods 1 and 3. Mean F1 moved from 0.7177 to 0.7157. We retain the rule because it restores known historical values. This comparison gives no evidence of an average predictive gain for logistic regression.

<a id="results"></a>

## 4.4 Eight models with and without client data

Each starting configuration gets the same two input sets, three periods and two thresholds. We fit once per period and reuse the probabilities for both thresholds. The reference also varies by period because the proportion of positive outcomes changes.

In [8]:
reference = [f1_score(y.loc[valid], [1] * int(valid.sum())) for _, valid in folds]
always_positive = pd.Series(reference, index=window_names, name='Always positive F1')
always_positive['Mean F1'] = always_positive.mean()
display(always_positive.round(4).to_frame().T)

def compare_settings(settings):
    rows = []
    for name, changes in settings:
        classifier = clone(models[name]).set_params(**changes)
        setting = ', '.join(f'{key}={value}' for key, value in changes.items()) or 'Starting'
        for feature_set, columns in feature_sets.items():
            scores = period_scores(classifier, purchases[columns])
            for threshold in thresholds:
                rows.append([name, setting, feature_set, threshold, *scores[threshold]])
    table = pd.DataFrame(rows, columns=['Model', 'Setting', 'Inputs', 'Threshold', *window_names])
    table['Mean F1'] = table[window_names].mean(axis=1)
    return table

starting = compare_settings([(name, {}) for name in models])
paired = starting.pivot(index=['Model', 'Threshold'], columns='Inputs', values='Mean F1')
paired['Change with clients'] = paired['Transactions + clients'] - paired['Transactions']
display(paired.round(4))

,Period 1,Period 2,Period 3,Mean F1
Always positive F1,0.7063,0.7464,0.6788,0.7105


Inputs                                 Transactions  Transactions + clients  \
Model                       Threshold                                         
Decision tree               0.45             0.7075                  0.7101   
                            0.50             0.7077                  0.7053   
Gaussian Naive Bayes        0.45             0.7252                  0.7160   
                            0.50             0.7258                  0.7166   
Gradient boosting           0.45             0.7406                  0.7333   
                            0.50             0.7263                  0.7240   
Histogram gradient boosting 0.45             0.7164                  0.7113   
                            0.50             0.7149                  0.7087   
K nearest neighbours        0.45             0.6950                  0.7155   
                            0.50             0.6950                  0.7155   
Logistic regression         0.45             0.7323                  0.7254   
                            0.50             0.7157                  0.7033   
Random forest               0.45             0.7416                  0.7454   
                            0.50             0.7352                  0.7355   
Small MLP                   0.45             0.6503                  0.6626   
                            0.50             0.6372                  0.6535   

Inputs                                 Change with clients  
Model                       Threshold                       
Decision tree               0.45                    0.0026  
                            0.50                   -0.0024  
Gaussian Naive Bayes        0.45                   -0.0092  
                            0.50                   -0.0092  
Gradient boosting           0.45                   -0.0073  
                            0.50                   -0.0023  
Histogram gradient boosting 0.45                   -0.0051  
                            0.50                   -0.0063  
K nearest neighbours        0.45                    0.0205  
                            0.50                    0.0205  
Logistic regression         0.45                   -0.0069  
                            0.50                   -0.0123  
Random forest               0.45                    0.0039  
                            0.50                    0.0003  
Small MLP                   0.45                    0.0122  
                            0.50                    0.0163

The always-positive reference has mean F1 of 0.7105, ranging from 0.6788 to 0.7464 as the repeat rate changes. In the original six-model comparison at 0.50, random forest led and gradient boosting on transactions was the next family. These were the first two families we tuned.

At 0.45 with starting settings, histogram boosting reaches 0.7164 using transactions and 0.7113 with client attributes. The small MLP starts at 0.6503 and 0.6626 respectively. Keeping the input set fixed lets us check next whether a change in leaf size or regularisation helps either model.

<a id="tuning"></a>

## 4.5 Comparing a small set of settings

The first six-model benchmark led us to tune random forest and gradient boosting. We then added histogram gradient boosting and a small MLP. We keep the earlier scores as a reference and use the same protocol throughout.

| Family | Settings compared | Reason |
|---|---|---|
| Random forest | Minimum leaf size 1, 5 or 10 | Larger leaves average more purchases. |
| Gradient boosting | 100 or 200 trees and learning rate 0.05 or 0.10 | A smaller update can need more trees. |
| Histogram gradient boosting | At most 31 or 15 leaves per tree | Fewer leaves limit how specific a tree can be. |
| MLP | L2 penalty `alpha` of 0.0001, 0.01 or 1 | Stronger regularisation penalises large weights. The 16-unit architecture stays fixed. |

The table below combines the starting scores with the new fits. For each input set and setting, the two thresholds score exactly the same purchases. The gain column measures the effect of changing the threshold alone.

In [9]:
forest_name = 'Random forest'
boost_name = 'Gradient boosting'
hist_name = 'Histogram gradient boosting'
mlp_name = 'Small MLP'
changes_to_try = [
    (forest_name, {'min_samples_leaf': 5}),
    (forest_name, {'min_samples_leaf': 10}),
    (boost_name, {'learning_rate': 0.05, 'n_estimators': 100}),
    (boost_name, {'learning_rate': 0.05, 'n_estimators': 200}),
    (boost_name, {'learning_rate': 0.10, 'n_estimators': 200}),
    (hist_name, {'max_leaf_nodes': 15}),
    (mlp_name, {'alpha': 0.01}),
    (mlp_name, {'alpha': 1.0}),
]
extra = compare_settings(changes_to_try)
results = pd.concat([starting, extra], ignore_index=True)
tuned_families = [forest_name, boost_name, hist_name, mlp_name]
tuning = results[results['Model'].isin(tuned_families)]
by_threshold = tuning.pivot(index=['Model', 'Inputs', 'Setting'], columns='Threshold', values='Mean F1')
by_threshold['Gain at 0.45'] = by_threshold[0.45] - by_threshold[0.50]
display(by_threshold[[0.50, 0.45, 'Gain at 0.45']].round(4))

Threshold                                                                                   0.5  \
Model                       Inputs                 Setting                                        
Gradient boosting           Transactions           Starting                              0.7263   
                                                   learning_rate=0.05, n_estimators=100  0.7271   
                                                   learning_rate=0.05, n_estimators=200  0.7220   
                                                   learning_rate=0.1, n_estimators=200   0.7202   
                            Transactions + clients Starting                              0.7240   
                                                   learning_rate=0.05, n_estimators=100  0.7365   
                                                   learning_rate=0.05, n_estimators=200  0.7227   
                                                   learning_rate=0.1, n_estimators=200   0.7103   
Histogram gradient boosting Transactions           Starting                              0.7149   
                                                   max_leaf_nodes=15                     0.7170   
                            Transactions + clients Starting                              0.7087   
                                                   max_leaf_nodes=15                     0.7152   
Random forest               Transactions           Starting                              0.7352   
                                                   min_samples_leaf=10                   0.7459   
                                                   min_samples_leaf=5                    0.7352   
                            Transactions + clients Starting                              0.7355   
                                                   min_samples_leaf=10                   0.7442   
                                                   min_samples_leaf=5                    0.7442   
Small MLP                   Transactions           Starting                              0.6372   
                                                   alpha=0.01                            0.6362   
                                                   alpha=1.0                             0.7221   
                            Transactions + clients Starting                              0.6535   
                                                   alpha=0.01                            0.6548   
                                                   alpha=1.0                             0.7173   

Threshold                                                                                  0.45  \
Model                       Inputs                 Setting                                        
Gradient boosting           Transactions           Starting                              0.7406   
                                                   learning_rate=0.05, n_estimators=100  0.7377   
                                                   learning_rate=0.05, n_estimators=200  0.7397   
                                                   learning_rate=0.1, n_estimators=200   0.7243   
                            Transactions + clients Starting                              0.7333   
                                                   learning_rate=0.05, n_estimators=100  0.7368   
                                                   learning_rate=0.05, n_estimators=200  0.7360   
                                                   learning_rate=0.1, n_estimators=200   0.7249   
Histogram gradient boosting Transactions           Starting                              0.7164   
                                                   max_leaf_nodes=15                     0.7234   
                            Transactions + clients Starting                              0.7113   
                                                   max_leaf_nodes=15                     0.7241   
Random forest               Transaction

### Reading the best scores

Within a model and input set, choose the highest mean F1 at full precision. An exact tie favours threshold 0.50 and then the first setting tried. Each row below shows the winning setting for that model and input set, across all three periods. A small client advantage for two separately tuned winners can include the effects of both tuning and client data. The paired tables above isolate the effect of changing one choice at a time.

In [10]:
ranked = results.sort_values(['Mean F1', 'Threshold'], ascending=False, kind='stable')
best = ranked.drop_duplicates(['Model', 'Inputs'])
display(best.reset_index(drop=True).round(4))

,Model,Setting,Inputs,Threshold,Period 1,Period 2,Period 3,Mean F1
0,Random forest,min_samples_leaf=5,Transactions + clients,0.45,0.7521,0.7431,0.7516,0.7489
1,Random forest,min_samples_leaf=10,Transactions,0.45,0.7433,0.7468,0.7484,0.7462
2,Gradient boosting,Starting,Transactions,0.45,0.7242,0.7433,0.7543,0.7406
3,Gradient boosting,"learning_rate=0.05, n_estimators=100",Transactions + clients,0.45,0.7312,0.7473,0.7320,0.7368
4,Small MLP,alpha=1.0,Transactions,0.45,0.7439,0.7327,0.7304,0.7357
5,Logistic regression,Starting,Transactions,0.45,0.7356,0.7422,0.7190,0.7323
6,Small MLP,alpha=1.0,Transactions + clients,0.45,0.7112,0.7376,0.7330,0.7272
7,Gaussian Naive Bayes,Starting,Transactions,0.50,0.7245,0.7448,0.7081,0.7258
8,Logistic regression,Starting,Transactions + clients,0.45,0.7173,0.7431,0.7159,0.7254
9,Histogram gradient boosting,max_leaf_nodes=15,Transactions + clients,0.45,0.7137,0.7331,0.7255,0.7241


**Random forest remains first** with customer attributes, five purchases per leaf and threshold 0.45. Its F1 is 0.7521, 0.7431 and 0.7516 across the three periods (mean 0.7489). In period 2, predicting a return for every purchase scores 0.7464. The forest leads the reference on average, with a small loss in that period. For gradient boosting the original 100 trees and 0.10 learning rate on transactions give the best mean (0.7406) at 0.45.

**The MLP is sensitive to regularisation.** On transactions at the same 0.45 threshold, increasing `alpha` from 0.0001 to 1 raises mean F1 from 0.6503 to 0.7357. Adding clients with the same settings gives 0.7272. Histogram boosting performs best with at most 15 leaves and threshold 0.45. Its mean is 0.7234 on transactions and 0.7241 with clients. That small difference gives little basis for a general claim about client attributes.

The Kaggle export below reproduces the previously submitted forest. The comparisons above use only the three internal periods.

<a id="kaggle-rf"></a>

## 4.6 Reproducing the random forest submission

On 20 November 2026 we submitted the forest with four client attributes, 100 trees, a minimum leaf size of 5 and threshold 0.45. Its public Kaggle F1 was **0.80798** (submission `58617907`). This cell preserves that exact configuration so the comparison with newer models is reproducible.

For this export we refit preprocessing and the forest on all 6,469 eligible labelled purchases. Their outcomes are known before the first test date. The 986 test purchases receive the same row-level corrections and customer join. The sample file supplies the IDs and output order. Running the cell writes a local CSV.

In [11]:
test = pd.read_csv(root / 'data' / 'test.csv')
sample = pd.read_csv(root / 'data' / 'sample_submission.csv')
assert purchases['purchase_date'].max() + horizon < pd.to_datetime(test['purchase_date']).min()
assert test['ID'].is_unique and sample['ID'].is_unique
assert set(test['ID']) == set(sample['ID']) and set(raw['ID']).isdisjoint(test['ID'])

future, _ = repair_history(test)
future[time_columns] = recover_time_intervals(future)
future = add_client_features(future)

columns = feature_sets['Transactions + clients']
forest = clone(models['Random forest']).set_params(min_samples_leaf=5)
submission_model = make_pipeline(clone(preparation), forest)
submission_model.fit(purchases[columns], y)
probability = submission_model.predict_proba(future[columns])[:, 1]
assert pd.Series(probability).between(0, 1).all()

labels = pd.Series((probability >= 0.45).astype(int), index=future['ID'])
submission = sample[['ID']].copy()
submission[target] = submission['ID'].map(labels)
assert submission.columns.equals(sample.columns) and submission[target].isin([0, 1]).all()
output_dir = root / 'outputs' / 'notebook04'
output_dir.mkdir(parents=True, exist_ok=True)
submission_file = output_dir / 'submission_rf_clients_leaf5_t045.csv'
submission.to_csv(submission_file, index=False)
pd.testing.assert_frame_equal(pd.read_csv(submission_file), submission)
print('Training purchases:', len(purchases))
print('Test predictions:', len(submission))
print(submission[target].value_counts().sort_index().to_string())
print('Saved:', submission_file.relative_to(root))

Training purchases: 6469
Test predictions: 986
repeat_purchase_90d
0    260
1    726
Saved: outputs/notebook04/submission_rf_clients_leaf5_t045.csv


### Scope of the results

Each purchase gets one vote in its period, so customers with many purchases contribute more observations. The same customer can appear on both sides of a cutoff. Performance for entirely new customers is a separate question.

The three internal periods were used to compare models, input sets, settings and thresholds. The best internal mean is therefore a selection score. Variation across the three periods shows how a result changed with time. The periods share much of their training data, so small differences cannot establish statistical significance. Giving periods equal weight also gives the shorter third period the same influence as each earlier year.

Earlier EDA examined target relationships and Notebook 03 used the later 1,942 purchases to compare logistic models. The internal comparison here fits and scores candidates entirely before that later block. We can use the block as a historical cross-check after fixing the model and threshold. Its purchases also enter the refit for the Kaggle submission, whose public score comes from different purchase occasions.

Course methods come from Week 04 (preprocessing and feature selection), Week 05 (assessment and metrics), Weeks 06 to 10 (the first five classifiers), Week 11 (boosting), Week 12 (the MLP) and Week 13 (regularisation). The three calendar windows, 90-day gap and same-row timing identity follow from the purchase task and the supplied data.